In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import arya
import pandas as pd

In [ ]:
from scipy.optimize import bisect

In [ ]:
from yasone import analysis as ana_utils
from yasone import selection as  filter_utils

In [ ]:
filter_utils.load_isochrones()

In [ ]:
from astropy import units as u

In [ ]:
from dataclasses import dataclass

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
from astropy.table import Table

In [ ]:
from yasone.photometry import outer_join_xmatch

import sys
sys.path.append("../photometry/")

from calc_panstarrs_offset import get_panstarrs

In [ ]:
from copy import copy

## Utilities

In [ ]:
def get_default_params(objname):
    obs_props = ana_utils.get_obs_props(objname)
    return filter_utils.filter_params(
        dm = obs_props["dm"], 
        iso_fe_h = obs_props["iso_fe_h"],
        iso_log_age = np.log10(obs_props["iso_age"]),
        objname = objname, 
        mode="gri", 
        E_BV = 0,
        r23_max_sigma=None,
        r_cen = 40/60,
        iso_width=0.15
    )
    

In [ ]:
def point_source_depth(mag_err_model, sigma):
    mag_err_0 = 2.5 / np.log(10) * 1/sigma

    return bisect(lambda x: mag_err_0 - mag_err_model(x), 10, 40)

In [ ]:
depth_sigma = 10

def get_g_depth(cat, params, sigma):
    g_err = ana_utils.fit_err(cat[params.gcol], cat[params.gcol + "_ERR"])
    g_depth = point_source_depth(g_err, sigma)
    return g_depth
    
def get_r_depth(cat, params, sigma):
    r_err = ana_utils.fit_err(cat[params.rcol], cat[params.rcol + "_ERR"])
    r_depth = point_source_depth(r_err, sigma)
    return r_depth

def get_i_depth(cat, params, sigma):
    i_err = ana_utils.fit_err(cat[params.icol], cat[params.icol + "_ERR"])
    i_depth = point_source_depth(i_err, sigma)
    return i_depth

In [ ]:
from dustmaps.sfd import SFDQuery

In [ ]:
def join_to_ps(cat, obj):
    cat_ps = get_panstarrs(obj)
    cat_ps = cat_ps[-cat_ps["iMeanKronMag"] + cat_ps["iMeanPSFMag"] < -0.02]
    cat_ps.rename_columns(
        ["G_MAG", "R_MAG", "I_MAG", "G_MAG_ERR", "R_MAG_ERR", "I_MAG_ERR", "ra", "dec", ],
        ["G_MAG_PS", "R_MAG_PS", "I_MAG_PS", "G_MAG_PS_ERR", "R_MAG_PS_ERR", "I_MAG_PS_ERR", "ra_ps", "dec_ps"]
    )


    cat = outer_join_xmatch(cat, cat_ps, ra1="ra", dec1="dec", ra2="ra_ps", dec2="dec_ps")
    return cat

In [ ]:
def get_bright_stars(cat_osiris, params, sat_g=18, sat_r=17.5, sat_i=18):
    filt = filter_utils.get_flags_filter(cat_osiris, params)
    
    cat_joined = join_to_ps(cat_osiris[filt], params.objname)
    
    filt = cat_joined["R_FLAGS"].mask # arbitrary column to exclude osiris

    cat = cat_joined[filt]
    
    cat["ra"] = cat["ra_ps"]
    cat["dec"] = cat["dec_ps"]
    coords = ana_utils.to_coords(cat)
    xi, eta = ana_utils.to_tangent(coords,  ana_utils.get_coord0(params.objname))
    cat["xi"] = xi * 60 * u.arcmin
    cat["eta"] = eta * 60 * u.arcmin

    E_BV = SFDQuery()(coords)
    A_g, A_r, A_i = ana_utils.get_extinction(E_BV)

    if "A_g" in cat.colnames:
        cat["G_MAG_PS"] -= A_g
        cat["R_MAG_PS"] -= A_r
        cat["I_MAG_PS"] -= A_i

    cat["G_MAG"] = cat["G_MAG_PS"]
    cat["R_MAG"] = cat["R_MAG_PS"]
    cat["I_MAG"] = cat["I_MAG_PS"]


    params_ps = copy(params)
    params_ps.gr_err = filter_utils.get_gr_err(cat_osiris, params)
    params_ps.ri_err=filter_utils.get_ri_err(cat_osiris, params)
    params_ps.detection_sigma = None
    
    subsets = filter_utils.select_subsets(cat, params_ps, )


    return cat, params_ps

## Plots

In [ ]:
ps_memb_kwargs = dict(
    s=9,
    lw=0.5, 
    color="w",
    ec="C3",
    label = "selected (PS)"
)

ps_all_kwargs = dict(
    s=3,
    lw=0.5, 
    color="none",
    ec="k",
    label = "all (PS)"

)

In [ ]:
def plot_gr_background_fancy(subsets, params, results, depth_sigma=5):
    filter_utils.plot_gr_background(subsets, params)
    cat = subsets["not_flagged"]
    plot_gr_depth(cat, params, sigma=depth_sigma)

    plt.ylim(27, 16)


In [ ]:
def plot_gr_centre_fancy(subsets, params, results, depth_sigma=5):
    filter_utils.plot_gr_centre(subsets, params)
    df = subsets["centre_all"]

    cat = subsets["not_flagged"]

    plot_gr_depth(cat, params, sigma=depth_sigma)
    plt.ylim(27, 16)


In [ ]:
def plot_ri_background_fancy(subsets, params, depth_sigma=5):
    filter_utils.plot_ri_background(subsets, params)
    cat = subsets["not_flagged"]

    plot_ri_depth(cat, params, sigma=depth_sigma)

    plt.ylim(27, 16)


In [ ]:
def plot_ri_centre_fancy(subsets, params, depth_sigma=5):
    filter_utils.plot_ri_centre(subsets, params)
    df = subsets["centre_all"]


    plot_ri_depth(cat, params, sigma=depth_sigma)
    plt.ylim(27, 16)


In [ ]:
def detection_plot(cat, params, depth_sigma=5):
    subsets = filter_utils.select_subsets(cat, params)
    results = filter_utils.catalog_stats(cat, params).iloc[0, :]

    fig, axs = plt.subplots(2, 3, figsize=(6, 4))

    plt.sca(axs[0][0])
    filter_utils.plot_tangent(subsets["not_flagged"], s=1, lw=0, color="k")
    plt.title("all")

    plt.sca(axs[1][0])
    filter_utils.plot_selected_points(subsets, params)
    
    plt.sca(axs[1][1])
    plot_gr_background_fancy(subsets, params, results, depth_sigma=depth_sigma)

    plt.sca(axs[0][1])
    plot_gr_centre_fancy(subsets, params, results, depth_sigma=depth_sigma)

    plt.sca(axs[1][2])
    plot_ri_background_fancy(subsets, params, depth_sigma=depth_sigma)

    plt.sca(axs[0][2])
    plot_ri_centre_fancy(subsets, params, depth_sigma=depth_sigma)

    plt.tight_layout()

    return fig, axs

In [ ]:
def plot_depth(r_depth, i_depth, sigma):
    x = np.linspace(-2, 2, 1000)
    mag_r = r_depth
    mag_i = np.minimum(i_depth, mag_r - x)
    mag_max = np.minimum(mag_r, mag_i +x)
    color=f"C0"
    plt.plot(x, mag_max, color=color)
    idx_0 = np.argmin(np.abs(x - -0.5))
    plt.annotate(xy=(x[idx_0], mag_max[idx_0]), text=f"${sigma} \\sigma$", color=color, fontsize=8,
                xytext=(0, 5), textcoords='offset points',)

def plot_ri_depth(cat, params, sigma):
    r_depth = get_r_depth(cat, params, sigma)
    i_depth = get_i_depth(cat, params, sigma)
    plot_depth(r_depth, i_depth, sigma)

def plot_gr_depth(cat, params, sigma):
    g_depth = get_g_depth(cat, params, sigma)
    r_depth = get_r_depth(cat, params, sigma)
    plot_depth(g_depth, r_depth, sigma)


In [ ]:
def overplot_panstarrs(axs, cat_ps, params):
    params_ps = copy(params)
    params_ps.flags_max = None
    params_ps.flags_iso_max = None
    params_ps.flas_weight_max = None

    subsets = filter_utils.select_subsets(cat_ps, params)
    plt.sca(axs[1][1])
    # plot_gr_background_fancy(subsets, params, depth_sigma=depth_sigma)
    df = subsets["background_all"]
    plt.scatter(df["G_MAG"] - df["R_MAG"], df["G_MAG"],  **ps_all_kwargs)
    df = subsets["background_selected"]
    plt.scatter(df["G_MAG"] - df["R_MAG"], df["G_MAG"],  **ps_memb_kwargs)

    plt.sca(axs[0][1])
    # plot_gr_centre_fancy(subsets, params, depth_sigma=depth_sigma)
    df = subsets["centre_all"]
    plt.scatter(df["G_MAG"] - df["R_MAG"], df["G_MAG"],  **ps_all_kwargs)
    df = subsets["centre_selected"]
    plt.scatter(df["G_MAG"] - df["R_MAG"], df["G_MAG"],  **ps_memb_kwargs)

    plt.sca(axs[1][2])
    # plot_ri_background_fancy(subsets, depth_sigma=depth_sigma)
    df = subsets["background_all"]
    plt.scatter(df["R_MAG"] - df["I_MAG"], df["R_MAG"],  **ps_all_kwargs)
    df = subsets["background_selected"]
    plt.scatter(df["R_MAG"] - df["I_MAG"], df["R_MAG"],  **ps_memb_kwargs)


    plt.sca(axs[0][2])
    # plot_ri_centre_fancy(subsets, depth_sigma=depth_sigma)
    df = subsets["centre_all"]
    plt.scatter(df["R_MAG"] - df["I_MAG"], df["R_MAG"],  **ps_all_kwargs)
    df = subsets["centre_selected"]
    plt.scatter(df["R_MAG"] - df["I_MAG"], df["R_MAG"],  **ps_memb_kwargs)


In [ ]:
def plot_significance_hist(cat, params):
    cat_filtered = filter_utils.filter_catalog(cat, params)
    coord0 = filter_utils.get_coord0(params.objname)

    region_mask, region_poly = filter_utils.get_region_mask(params.objname)
    safe_region = filter_utils.get_sample_region(params.r_cen, region_mask, region_poly,
                                    coord0)

    background_counts = [filter_utils.rand_overdensity(cat_filtered, params.r_cen, safe_region=safe_region) for i in range(10_000)]
    Ncen = filter_utils.count_centre(cat_filtered, params.r_cen)

    plt.hist(background_counts, histtype="step", bins=-0.5 + np.arange(0, np.max(background_counts)))
    plt.gca().set_box_aspect(1)
    plt.xlabel("N within random circle")
    plt.ylabel("counts")
    p = np.mean(Ncen <= background_counts)
    plt.text(0.9, 0.9, f"p = {p}", transform=plt.gca().transAxes, fontsize=8, ha="right", va="bottom", color="C0")

    plt.axvline(Ncen, color="k")
    plt.xlim(-0.5)

In [ ]:
def plot_contour(cat, params, xymax=5, N=100, vmin=None, vmax=None):
    cat_filtered = filter_utils.filter_catalog(cat, params)

    x = np.linspace(np.min(cat["xi"]), np.max(cat["xi"]), N)
    y = np.linspace(np.min(cat["eta"]), np.max(cat["eta"]), N)

    xgrid, ygrid = np.meshgrid(x, y)
    counts = np.zeros((N, N))

    for i in range(N):
        for j in range(N):
            counts[i, j] = filter_utils.count_centre(cat_filtered, params.r_cen, xi0=xgrid[i, j], eta0=ygrid[i, j])

    level_step = np.maximum(1, np.ceil(np.max(counts) / 15))

    p = plt.contourf(x, y, counts, levels=np.arange(np.min(counts), np.max(counts)+level_step, level_step), vmin=vmin, vmax=vmax)

    ax =  plt.gca()
    plt.colorbar(p, label="counts within circle", fraction=0.046, pad=0.04)
    plt.xlabel(r"$\xi$ / arcmin")
    plt.ylabel(r"$\eta$ / arcmin")

    plt.scatter(0, 0, color="k", s=0.3, lw=0)
    ax.set_aspect(1)
    ax.invert_xaxis()
    return counts


# Yasone 1

In [ ]:
params = get_default_params("yasone1")
params.xi = 3.5
params.eta = 2.5

cat = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift", deredden=True)



In [ ]:
plot_contour(cat, params)

In [ ]:
plot_significance_hist(cat, params)

In [ ]:
cat_ps, params_ps = get_bright_stars(cat, params)

In [ ]:
fig, axs = detection_plot(cat, params,)
overplot_panstarrs(axs, cat_ps, params_ps)

In [ ]:
detection_plot(cat, params)
overplot_panstarrs(axs, cat_ps, params_ps)
arya.Legend(-1)

In [ ]:
# cat_forced = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced", shiftname="allcolours_panstarrs_shift", deredden=True)


In [ ]:
# cat_psf = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_psf", shiftname="allcolours_panstarrs_shift", deredden=True)


In [ ]:
# filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_LB_3")
# filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_3")
# filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF")


In [ ]:
# for filt in ["G", "R", "I"]:
#     cat_psf[f"{filt}_MAG"] = cat_psf[f"{filt}_MED_MAG_PSF"]
#     cat_psf[f"{filt}_MAG_ERR"] = cat_psf[f"{filt}_MED_MAG_PSF_ERR"]
    
# cat_ps_psf = correct_bright_stars(cat_psf, "yasone1", sat_r = 20)

In [ ]:
# detection_plot()
# fig = plt.gcf()
# plt.sca(fig.axes[1])
# plt.scatter(ps_julen["G_MAG"] - ps_julen["R_MAG"] - A_g + A_r, ps_julen["G_MAG"] - A_g, alpha=0.3)

# plt.sca(fig.axes[2])
# plt.scatter(ps_julen["R_MAG"] - ps_julen["I_MAG"] - A_r + A_i, ps_julen["R_MAG"] - A_r, alpha=0.3)


In [ ]:
filter_utils.plot_with_hist(cat, params)

Correspondance
- 123802655211789538: row 31 (Gaia)
- 123802655213495679: row 32 (Gaia)
- 123812655293951255: row 33 (Gaia)
- 123802655208552790: row 34 ?? (PanSTARRS)

In [ ]:
ps = get_panstarrs("yasone1")

In [ ]:
ps_julen = ps[np.isin(ps["ObjID"], [123802655211789538, 123802655213495679, 123812655293951255, 123802655208552790])
    ]

In [ ]:
ps_julen["rMeanKronMag"] - ps_julen["rMeanPSFMag"]

In [ ]:
A_g, A_r, A_i = np.nanmedian(cat_ps[["A_g", "A_r", "A_i"]].to_pandas().to_numpy(), axis=0)

In [ ]:
A_g, A_r, A_i

In [ ]:
detection_plot(cat_ps_psf, params, xi=-2/60, eta=-2/60)

In [ ]:
cat_ps_lb = cat_ps.copy()
cat_ps_lb["G_MAG"] = cat_ps_lb["G_MED_MAG_APER_LB_3"]
cat_ps_lb["R_MAG"] = cat_ps_lb["R_MED_MAG_APER_LB_3"]
cat_ps_lb["I_MAG"] = cat_ps_lb["I_MED_MAG_APER_LB_3"]


In [ ]:
detection_plot(cat_ps_lb, params, xi=-1/60, eta=-2/60)

In [ ]:
params.gcol = "G_MED_MAG_APER_3"
params.rcol = "R_MED_MAG_APER_3"
params.icol = "I_MED_MAG_APER_3"
detection_plot(cat_forced, params, xi=-1/60, eta=-2/60)

# Yasone 2

In [ ]:
params = get_default_params("yasone2")
params.eta = 0.5
params.xi = 3 

In [ ]:
cat = ana_utils.read_catalogue("yasone2", filter_bad=False, deredden=True, catname="allcolours", shiftname="allcolours_panstarrs_shift")
cat_ps, params_ps = get_bright_stars(cat, params)

In [ ]:
plot_contour(cat, params)

In [ ]:
plot_significance_hist(cat, params)

In [ ]:
fig, axs = detection_plot(cat, params,)
overplot_panstarrs(axs, cat_ps, params_ps)

In [ ]:
cat_psf = ana_utils.read_catalogue("yasone2", filter_bad=False, deredden=True, catname="allcolours_forced_psf", shiftname="allcolours_panstarrs_shift")
cat_forced = ana_utils.read_catalogue("yasone2", filter_bad=False, deredden=True, catname="allcolours_forced", shiftname="allcolours_panstarrs_shift")


In [ ]:
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF")
for filt in ["G", "R", "I"]:
    cat_psf[f"{filt}_MAG"] = cat_psf[f"{filt}_MED_MAG_PSF"]
    cat_psf[f"{filt}_MAG_ERR"] = cat_psf[f"{filt}_MED_MAG_PSF_ERR"]
    
cat_ps_psf = correct_bright_stars(cat_psf, "yasone2", sat_r = 20)

In [ ]:
filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_LB_3")
filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_3")


In [ ]:
cat["xi"] -= 0.0
cat["eta"] -= 0

In [ ]:
plt.scatter(cat_ps["G_MAG"], cat_ps["G_MAG_PS"] - cat_ps["G_MAG"])
plt.scatter(cat_ps["R_MAG"], cat_ps["R_MAG_PS"] - cat_ps["R_MAG"])
plt.scatter(cat_ps["I_MAG"], cat_ps["I_MAG_PS"] - cat_ps["I_MAG"])

In [ ]:
params.gcol

In [ ]:
filter_utils.plot_with_hist(cat, params)

In [ ]:
detection_plot(cat_ps_psf, params, xi=3/60, eta=0.5/60)

In [ ]:
params.gcol = "G_MED_MAG_APER_LB_3"
params.rcol = "R_MED_MAG_APER_LB_3"
params.icol = "I_MED_MAG_APER_LB_3"
detection_plot(cat, params, xi=3/60, eta=0.5/60)

# Yasone 3

In [ ]:
params = get_default_params("yasone3")
params.A_i_extra = -0.1
params.dm -= 0.5
params.xi = 1
params.eta = -2.5

In [ ]:
params.r_cen = 40/60

In [ ]:
plot_contour(cat, params)

In [ ]:
plot_significance_hist(cat, params)

In [ ]:
cat = ana_utils.read_catalogue("yasone3", filter_bad=False, catname="allcolours", deredden=True)

In [ ]:
cat_ps, params_ps = get_bright_stars(cat, params)

In [ ]:
fig, axs = detection_plot(cat, params,)
overplot_panstarrs(axs, cat_ps, params_ps)

In [ ]:
cat_psf = ana_utils.read_catalogue("yasone3", filter_bad=False, catname="allcolours_forced_psf", shiftname="allcolours_panstarrs_shift", deredden=True)

In [ ]:
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF")
for filt in ["G", "R", "I"]:
    cat_psf[f"{filt}_MAG"] = cat_psf[f"{filt}_MED_MAG_PSF"]
    cat_psf[f"{filt}_MAG_ERR"] = cat_psf[f"{filt}_MED_MAG_PSF_ERR"]
    
cat_ps_psf = correct_bright_stars(cat_psf, "yasone3", sat_r = 20)

In [ ]:
detection_plot(cat_ps_psf, params, xi=1.0/60, eta=-2.5/60)

In [ ]:
filter_utils.plot_with_hist(cat, params)

# Extra analysis plots

In [ ]:
params = get_default_params("yasone2")
params.xi = -2
params.eta = -3
cat = ana_utils.read_catalogue("yasone2", filter_bad=False, deredden=True, catname="allcolours", shiftname="allcolours_panstarrs_shift")
cat_ps, params_ps = get_bright_stars(cat, params)

In [ ]:
mag_diff_thresh = filter_utils.derive_threshhold(cat["MAG_23"], cat)

In [ ]:
cat_star = cat[(cat["MAG_23"] < mag_diff_thresh) & filter_utils.get_flags_filter(cat, params)]
cat_gal = cat[(cat["MAG_23"] > mag_diff_thresh) & filter_utils.get_flags_filter(cat, params)]

plt.scatter(cat_star["xi"], cat_star["eta"], s=1, label="star?")
plt.scatter(cat_gal["xi"], cat_gal["eta"], s=1, label="galaxy?")
cat_bright = cat_ps[cat_ps["R_MAG"] < 16]
plt.scatter(cat_bright["xi"], cat_bright["eta"], label="bright", s=10)

filter_utils.tangent_axis()
plt.xlim(3, -3)
plt.ylim(-3, 3)
arya.Legend(-1)

In [ ]:
params = get_default_params("yasone1")
params.xi = -2
params.eta = -3
cat = ana_utils.read_catalogue("yasone2", filter_bad=False, deredden=True, catname="allcolours", shiftname="allcolours_panstarrs_shift")


In [ ]:
filters = ["g", "r", "i"]

fig, axs = plt.subplots(3, 2, sharex=True, sharey="col", figsize=(6, 3))

for i, filt in enumerate(filters):
    plt.sca(axs[i][0])
    plt.scatter(cat_good[f"{filt.upper()}_MAG"], cat_good[f"{filt.upper()}_MAG_ERR"])
    if filt == "g":
        mag5 = get_g_depth(cat_good, params, 5)
    elif filt == "r":
        mag5 = get_r_depth(cat_good, params, 5)
    elif filt == "i":
        mag5 = get_i_depth(cat_good, params, 5)

    plt.scatter(mag5, 0.2, s=10)
    plt.axvline(mag5, color="C1")

    plt.ylim(0, 1)
    plt.ylabel(f"$\\delta {filt}$")

    if i == 2:
        plt.xlabel("magnitude")

    
    plt.sca(axs[i][1])
    plt.hist(cat_good[f"{filt.upper()}_MAG"])
    plt.axvline(mag5, color="C1")
    plt.text(mag5, 0, r"$5\sigma$", ha="right", color="C1")

    if i == 2:
        plt.xlabel("magnitude")

plt.xlim(16, 27)

cat_good = cat[filter_utils.get_flags_filter(cat, params)]

# Yasone 1 Extra

In [ ]:
plt.scatter(cat_psf["G_MED_MAG_PSF"], cat_psf["G_MED_MAG_PSF_ERR"], s=1)
plt.scatter(cat_psf["G_MAG"], cat_psf["G_MAG_ERR"], s=1)
plt.ylim(0, 1)

gr_err = ana_utils.fit_err(cat_psf["G_MAG"], cat_psf["G_MAG_ERR"])
x = np.linspace(20, 26, 1000)
plt.plot(x, gr_err(x))
plt.xlabel("g mag")
plt.ylabel("g mag err")

In [ ]:
cat_julen_sep = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="julen")

params_sep = copy(params)
params_sep.flags_max = None
params_sep.flags_weight_max = None
params_sep.r23_max_sigma = None
params_sep.A_V = 0.2
params_sep.A_i_extra += 0.1
params_sep.dm -= -0.1


In [ ]:
detection_plot(cat_julen_sep, params_sep)

In [ ]:
params_psf = copy(params)
params_psf.rcol = "R_MAG_PSF"
params_psf.gcol = "G_MAG_PSF"
params_psf.icol = "I_MAG_PSF"
params_psf.mode = "ri"


params_sep = copy(params)
params_sep.flags_max = None
params_sep.flags_weight_max = None
params_sep.r23_max_sigma = None


In [ ]:
cat = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours")
cat_psf = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_psf")

cat_julen = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_julen_stack")
cat_julen_sep = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="julen")

In [ ]:
filter_utils.calibrate_mag_col(cat_psf, "MAG_PSF")

In [ ]:
for filt in ["G", "R", "I"]:
    cat_psf.rename_column(f"{filt}_MAGERR_PSF", f"{filt}_MAG_PSF_ERR")

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(4, 4), sharex=True, sharey=True)

plt.sca(axs[0][0])
subsets = filter_utils.select_subsets(cat, params)
filter_utils.plot_ri_centre(subsets, params)
plt.title("defaults")



plt.sca(axs[1][0])
subsets = filter_utils.select_subsets(cat_psf, params_psf, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_ri_centre(subsets, params_psf)
plt.title("psf mag")

plt.sca(axs[0][1])
subsets = filter_utils.select_subsets(cat_julen, params, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_ri_centre(subsets, params)

plt.title("julen image")


plt.sca(axs[1][1])
subsets = filter_utils.select_subsets(cat_julen_sep, params_sep, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_ri_centre(subsets, params_sep)

plt.title("julen + SEP")

plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(4, 4), sharex=True, sharey=True)

plt.sca(axs[0][0])
subsets = filter_utils.select_subsets(cat, params, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_gr_centre(subsets, params)
plt.title("defaults")



plt.sca(axs[1][0])
subsets = filter_utils.select_subsets(cat_psf, params_psf, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_gr_centre(subsets, params_psf)
plt.title("psf mag")

plt.sca(axs[0][1])
subsets = filter_utils.select_subsets(cat_julen, params, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_gr_centre(subsets, params)

plt.title("julen image")


plt.sca(axs[1][1])
subsets = filter_utils.select_subsets(cat_julen_sep, params_sep, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_gr_centre(subsets, params_sep)

plt.title("julen + SEP")

plt.tight_layout()

In [ ]:
params = get_default_params("yasone1")
params.mode = "ri"
filter_utils.plot_with_hist(cat, params)

In [ ]:
params = get_default_params("yasone1")
filter_utils.plot_with_hist(cat, params)

In [ ]:
params

In [ ]:
params_new = copy(params)
params_new.A_V += 0.3
params_new.dm -= 0.3
params_new.mode = "ri"
filter_utils.plot_with_hist(cat, params_new)

In [ ]:
params_new.mode = "gri"
filter_utils.plot_with_hist(cat, params_new)